In [ ]:
import math
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, log_loss, brier_score_loss

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
batch_size = 32
labels = np.array(dataset["label"])
predictions = []
confidences = []
positive_probs = []
full_probs = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        confs = probs.max(dim=-1).values

    predictions.extend(preds.cpu().tolist())
    confidences.extend(confs.cpu().tolist())
    positive_probs.extend(probs[:, 1].cpu().tolist())
    full_probs.extend(probs.cpu().tolist())

predictions = np.array(predictions)
confidences = np.array(confidences)
positive_probs = np.array(positive_probs)
full_probs = np.array(full_probs)

print(f"Completed inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
cm = confusion_matrix(labels, predictions)
nll = log_loss(labels, full_probs, labels=[0, 1])
brier = brier_score_loss(labels, positive_probs)

correct = (predictions == labels).astype(int)
n_bins = 10
bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
ece = 0.0
bin_summaries = []

for i in range(n_bins):
    left = bin_edges[i]
    right = bin_edges[i + 1]
    if i == n_bins - 1:
        mask = (confidences >= left) & (confidences <= right)
    else:
        mask = (confidences >= left) & (confidences < right)

    count = int(mask.sum())
    if count > 0:
        bin_acc = float(correct[mask].mean())
        bin_conf = float(confidences[mask].mean())
        gap = abs(bin_acc - bin_conf)
        ece += (count / len(labels)) * gap
        bin_summaries.append({
            "bin": i,
            "range": f"[{left:.1f}, {right:.1f}]" if i == n_bins - 1 else f"[{left:.1f}, {right:.1f})",
            "count": count,
            "avg_confidence": bin_conf,
            "accuracy": bin_acc,
            "gap": gap,
        })
    else:
        bin_summaries.append({
            "bin": i,
            "range": f"[{left:.1f}, {right:.1f}]" if i == n_bins - 1 else f"[{left:.1f}, {right:.1f})",
            "count": 0,
            "avg_confidence": None,
            "accuracy": None,
            "gap": None,
        })

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"NLL      : {nll:.4f}")
print(f"Brier    : {brier:.4f}")
print(f"ECE      : {ece:.4f}")
print("Confusion matrix:")
print(cm)

print("\nCalibration bin summary:")
for summary in bin_summaries:
    if summary["count"] == 0:
        print(f"bin={summary['bin']:2d} range={summary['range']:>11} count=0")
    else:
        print(
            f"bin={summary['bin']:2d} range={summary['range']:>11} count={summary['count']:3d} "
            f"avg_conf={summary['avg_confidence']:.4f} acc={summary['accuracy']:.4f} gap={summary['gap']:.4f}"
        )

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
correct_indices = np.where(predictions == labels)[0]
incorrect_indices = np.where(predictions != labels)[0]

lowest_conf_correct_idx = correct_indices[np.argmin(confidences[correct_indices])] if len(correct_indices) > 0 else None
highest_conf_incorrect_idx = incorrect_indices[np.argmax(confidences[incorrect_indices])] if len(incorrect_indices) > 0 else None

def print_example(idx, title):
    row = dataset[int(idx)]
    true_label = int(labels[idx])
    pred_label = int(predictions[idx])
    conf = float(confidences[idx])
    pos_prob = float(positive_probs[idx])
    neg_prob = float(full_probs[idx][0])
    print(title)
    print(f"index      : {idx}")
    print(f"sentence1  : {row['sentence1']}")
    print(f"sentence2  : {row['sentence2']}")
    print(f"true label : {true_label} ({label_map[true_label]})")
    print(f"pred label : {pred_label} ({label_map[pred_label]})")
    print(f"confidence : {conf:.4f}")
    print(f"p(class=0) : {neg_prob:.4f}")
    print(f"p(class=1) : {pos_prob:.4f}")
    print("-" * 80)

if lowest_conf_correct_idx is not None:
    print_example(lowest_conf_correct_idx, "Lowest-confidence correct example")
else:
    print("No correct predictions found.")

if highest_conf_incorrect_idx is not None:
    print_example(highest_conf_incorrect_idx, "Highest-confidence incorrect example")
else:
    print("No incorrect predictions found.")

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"nll={nll:.4f}")
print(f"brier={brier:.4f}")
print(f"ece={ece:.4f}")